# Day 074 | Autoencoders

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning

**Author:** Ripon Chandra Malo

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Encoder | Compresses input into a low-dimensional latent code | Summarizing 10 soil properties into 2 key factors |
| Decoder | Reconstructs input from the latent code | Recovering full soil profile from compressed form |
| Latent space | Compact internal representation | Hidden structure in soil classification data |
| Reconstruction error | Difference between input and output | Detecting unusual or erroneous soil samples |
| Anomaly detection | Flagging samples with high reconstruction error | Identifying mislabeled or contaminated samples |

---

## Our Geotechnical Problem

A geotechnical lab has processed **hundreds of soil samples** from a large site investigation.
Each sample has 8 measured properties (moisture, density, plasticity, etc.).

We need to:
1. Find a compact representation of the soil data (dimensionality reduction)
2. **Detect anomalous samples** — possibly mislabeled, contaminated, or from unexpected strata
3. Visualize the latent structure of soil types

---

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## Part 1: Generating Synthetic Soil Lab Data

We simulate a dataset with three soil types (clay, silt, sand), each having
correlated properties. A few samples are intentionally anomalous.

| Property | Unit | Typical Range |
|----------|------|---------------|
| Moisture content | % | 10-60 |
| Dry density | kN/m3 | 12-20 |
| Liquid limit | % | 20-80 |
| Plastic limit | % | 10-40 |
| Fines content | % | 5-95 |
| SPT N-value | - | 2-50 |
| Compression index | - | 0.05-0.8 |
| Undrained shear strength | kPa | 10-200 |

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE SYNTHETIC SOIL LAB DATA
# ───────────────────────────────────────────────────────

def generate_soil_data(n_normal=300, n_anomaly=15, seed=42):
    """
    Generate synthetic soil laboratory data with three soil types
    plus anomalous samples.
    
    Features: [moisture, dry_density, LL, PL, fines, SPT_N, Cc, Su]
    """
    np.random.seed(seed)
    
    # Soil type 1: Soft Clay
    n1 = n_normal // 3
    clay = np.column_stack([
        np.random.normal(45, 8, n1),    # moisture
        np.random.normal(14, 1.0, n1),  # dry density
        np.random.normal(60, 10, n1),   # LL
        np.random.normal(28, 5, n1),    # PL
        np.random.normal(85, 5, n1),    # fines
        np.random.normal(5, 2, n1),     # SPT N
        np.random.normal(0.5, 0.1, n1), # Cc
        np.random.normal(30, 10, n1),   # Su
    ])
    
    # Soil type 2: Silt
    n2 = n_normal // 3
    silt = np.column_stack([
        np.random.normal(30, 5, n2),
        np.random.normal(16, 1.0, n2),
        np.random.normal(35, 6, n2),
        np.random.normal(22, 4, n2),
        np.random.normal(65, 8, n2),
        np.random.normal(15, 5, n2),
        np.random.normal(0.2, 0.05, n2),
        np.random.normal(80, 20, n2),
    ])
    
    # Soil type 3: Sand
    n3 = n_normal - n1 - n2
    sand = np.column_stack([
        np.random.normal(15, 3, n3),
        np.random.normal(18, 0.8, n3),
        np.random.normal(0, 2, n3),     # non-plastic (near 0)
        np.random.normal(0, 1, n3),     # non-plastic
        np.random.normal(12, 5, n3),
        np.random.normal(30, 8, n3),
        np.random.normal(0.05, 0.02, n3),
        np.random.normal(0, 5, n3),     # no undrained strength for sand
    ])
    
    # Normal data
    normal_data = np.vstack([clay, silt, sand])
    labels_normal = np.array([0]*n1 + [1]*n2 + [2]*n3)  # 0=clay, 1=silt, 2=sand
    
    # Anomalous samples (physically inconsistent)
    anomalies = np.column_stack([
        np.random.uniform(5, 80, n_anomaly),     # extreme moisture
        np.random.uniform(10, 22, n_anomaly),    # inconsistent density
        np.random.uniform(0, 100, n_anomaly),    # random LL
        np.random.uniform(0, 60, n_anomaly),     # random PL
        np.random.uniform(0, 100, n_anomaly),    # random fines
        np.random.uniform(0, 60, n_anomaly),     # random SPT
        np.random.uniform(0, 1.0, n_anomaly),    # random Cc
        np.random.uniform(0, 300, n_anomaly),    # random Su
    ])
    
    all_data = np.vstack([normal_data, anomalies])
    is_anomaly = np.array([False]*n_normal + [True]*n_anomaly)
    labels = np.concatenate([labels_normal, np.full(n_anomaly, -1)])
    
    # Clip to realistic ranges
    all_data = np.clip(all_data, 0, None)
    
    return all_data, is_anomaly, labels

feature_names = ['Moisture (%)', 'Dry Density', 'LL (%)', 'PL (%)',
                 'Fines (%)', 'SPT N', 'Cc', 'Su (kPa)']

data, is_anomaly, labels = generate_soil_data(n_normal=300, n_anomaly=15)
print(f"Total samples: {len(data)}")
print(f"Normal: {(~is_anomaly).sum()}, Anomalous: {is_anomaly.sum()}")
print(f"Features: {len(feature_names)}")

In [ ]:
# ───────────────────────────────────────────────────────
# NORMALIZE AND PREPARE DATA
# ───────────────────────────────────────────────────────

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Use only normal samples for training (anomalies should have high error)
train_data = data_scaled[~is_anomaly]
np.random.shuffle(train_data)

n_train = int(len(train_data) * 0.8)
X_train = torch.FloatTensor(train_data[:n_train])
X_val   = torch.FloatTensor(train_data[n_train:])
X_all   = torch.FloatTensor(data_scaled)

train_ds = TensorDataset(X_train, X_train)  # autoencoder: input == target
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"Training: {X_train.shape[0]} normal samples")
print(f"Validation: {X_val.shape[0]} normal samples")
print(f"Full set (for anomaly detection): {X_all.shape[0]} samples")

## Part 2: Building the Autoencoder

An autoencoder has two parts:

```
Input (8) --> Encoder --> Latent (2) --> Decoder --> Output (8)
              [8->16->8->2]              [2->8->16->8]
```

The **bottleneck** (latent dimension = 2) forces the network to learn a compressed
representation. Normal soil samples can be reconstructed well; anomalous samples cannot.

---

In [ ]:
# ───────────────────────────────────────────────────────
# AUTOENCODER ARCHITECTURE
# ───────────────────────────────────────────────────────

class SoilAutoencoder(nn.Module):
    """Autoencoder for soil property data."""
    
    def __init__(self, input_dim=8, latent_dim=2):
        super().__init__()
        
        # Encoder: compress 8 features to latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim)
        )
        
        # Decoder: reconstruct from latent_dim to 8 features
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

model = SoilAutoencoder(input_dim=8, latent_dim=2)
n_params = sum(p.numel() for p in model.parameters())
print(f"Autoencoder parameters: {n_params:,}")
print(f"\nArchitecture:\n{model}")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN THE AUTOENCODER
# ───────────────────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

epochs = 200
train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for xb, _ in train_dl:
        x_hat, z = model(xb)
        loss = criterion(x_hat, xb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_dl))
    
    model.eval()
    with torch.no_grad():
        val_hat, _ = model(X_val)
        val_loss = criterion(val_hat, X_val).item()
        val_losses.append(val_loss)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train: {train_losses[-1]:.6f}  Val: {val_losses[-1]:.6f}")

print(f"\nFinal — Train: {train_losses[-1]:.6f}  Val: {val_losses[-1]:.6f}")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING CURVES
# ───────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Training', linewidth=2)
ax.plot(val_losses, label='Validation', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (Reconstruction Error)')
ax.set_title('Autoencoder Training  |  Soil Property Reconstruction')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Anomaly Detection via Reconstruction Error

The key idea: the autoencoder is trained on **normal** soil samples only.

- **Normal samples** are reconstructed well (low error)
- **Anomalous samples** are reconstructed poorly (high error)

We set a **threshold** on reconstruction error to flag anomalies.

---

In [ ]:
# ───────────────────────────────────────────────────────
# COMPUTE RECONSTRUCTION ERROR FOR ALL SAMPLES
# ───────────────────────────────────────────────────────

model.eval()
with torch.no_grad():
    X_hat, Z = model(X_all)

# Per-sample reconstruction error (MSE)
recon_error = ((X_all - X_hat) ** 2).mean(dim=1).numpy()

# Statistics for normal vs anomalous
normal_errors = recon_error[~is_anomaly]
anomaly_errors = recon_error[is_anomaly]

print(f"Normal samples  — Mean error: {normal_errors.mean():.4f}, Max: {normal_errors.max():.4f}")
print(f"Anomaly samples — Mean error: {anomaly_errors.mean():.4f}, Min: {anomaly_errors.min():.4f}")

In [ ]:
# ───────────────────────────────────────────────────────
# RECONSTRUCTION ERROR DISTRIBUTION
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(normal_errors, bins=30, alpha=0.7, color='#2980b9', label='Normal', density=True)
axes[0].hist(anomaly_errors, bins=10, alpha=0.7, color='#e74c3c', label='Anomaly', density=True)

# Threshold: mean + 3*std of normal errors
threshold = normal_errors.mean() + 3 * normal_errors.std()
axes[0].axvline(x=threshold, color='k', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.3f}')

axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sorted error plot
sorted_idx = np.argsort(recon_error)
axes[1].bar(range(len(recon_error)), recon_error[sorted_idx],
           color=['#e74c3c' if is_anomaly[i] else '#2980b9' for i in sorted_idx],
           alpha=0.7, width=1.0)
axes[1].axhline(y=threshold, color='k', linestyle='--', linewidth=2)
axes[1].set_xlabel('Sample (sorted by error)')
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('All Samples Sorted by Reconstruction Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detection metrics
detected = recon_error > threshold
tp = (detected & is_anomaly).sum()
fp = (detected & ~is_anomaly).sum()
fn = (~detected & is_anomaly).sum()
tn = (~detected & ~is_anomaly).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\nAnomaly Detection Results (threshold = {threshold:.4f})")
print(f"True Positives:  {tp}  |  False Positives: {fp}")
print(f"True Negatives:  {tn}  |  False Negatives: {fn}")
print(f"Precision: {precision:.3f}  |  Recall: {recall:.3f}")

## Part 4: Latent Space Visualization

Since our latent dimension is 2, we can directly plot the encoded representations.
This shows how the autoencoder has learned to cluster different soil types.

---

In [ ]:
# ───────────────────────────────────────────────────────
# LATENT SPACE VISUALIZATION
# ───────────────────────────────────────────────────────

Z_np = Z.numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by soil type
soil_names = ['Clay', 'Silt', 'Sand', 'Anomaly']
soil_colors = ['#e74c3c', '#f39c12', '#27ae60', '#000000']

for i, (name, color) in enumerate(zip(soil_names[:3], soil_colors[:3])):
    mask = labels == i
    axes[0].scatter(Z_np[mask, 0], Z_np[mask, 1], c=color, label=name, alpha=0.6, s=30)

mask_anom = labels == -1
axes[0].scatter(Z_np[mask_anom, 0], Z_np[mask_anom, 1], c='k', marker='x',
               s=80, linewidth=2, label='Anomaly')

axes[0].set_xlabel('Latent Dimension 1')
axes[0].set_ylabel('Latent Dimension 2')
axes[0].set_title('Latent Space  |  Colored by Soil Type')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Color by reconstruction error
sc = axes[1].scatter(Z_np[:, 0], Z_np[:, 1], c=recon_error, cmap='hot_r',
                    alpha=0.7, s=30)
axes[1].scatter(Z_np[mask_anom, 0], Z_np[mask_anom, 1], facecolors='none',
               edgecolors='blue', s=100, linewidth=2, label='Anomaly')

plt.colorbar(sc, ax=axes[1], label='Reconstruction Error')
axes[1].set_xlabel('Latent Dimension 1')
axes[1].set_ylabel('Latent Dimension 2')
axes[1].set_title('Latent Space  |  Colored by Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# COMPARE AUTOENCODER vs PCA FOR DIMENSIONALITY REDUCTION
# ───────────────────────────────────────────────────────

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
Z_pca = pca.fit_transform(data_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (name, color) in enumerate(zip(soil_names[:3], soil_colors[:3])):
    mask = labels == i
    axes[0].scatter(Z_pca[mask, 0], Z_pca[mask, 1], c=color, label=name, alpha=0.6, s=30)
axes[0].scatter(Z_pca[mask_anom, 0], Z_pca[mask_anom, 1], c='k', marker='x', s=80, linewidth=2, label='Anomaly')
axes[0].set_title('PCA (Linear)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

for i, (name, color) in enumerate(zip(soil_names[:3], soil_colors[:3])):
    mask = labels == i
    axes[1].scatter(Z_np[mask, 0], Z_np[mask, 1], c=color, label=name, alpha=0.6, s=30)
axes[1].scatter(Z_np[mask_anom, 0], Z_np[mask_anom, 1], c='k', marker='x', s=80, linewidth=2, label='Anomaly')
axes[1].set_title('Autoencoder (Non-linear)')
axes[1].set_xlabel('Latent 1')
axes[1].set_ylabel('Latent 2')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Dimensionality Reduction: PCA vs Autoencoder', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# FEATURE-WISE RECONSTRUCTION QUALITY
# ───────────────────────────────────────────────────────

model.eval()
with torch.no_grad():
    X_normal = torch.FloatTensor(data_scaled[~is_anomaly])
    X_hat_normal, _ = model(X_normal)

feature_errors = ((X_normal - X_hat_normal) ** 2).mean(dim=0).numpy()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(feature_names)), feature_errors, color='#2980b9', alpha=0.8)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=30, ha='right')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Reconstruction Error by Feature  |  Normal Samples Only')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Practice Exercises

Try these exercises to deepen your understanding:

```python
# Exercise 1: Deeper autoencoder
# Add more layers to the encoder/decoder.
# Does a deeper network improve reconstruction or anomaly detection?

# Exercise 2: Different latent dimensions
# Try latent_dim = 1, 3, 4, 8 (same as input). What happens?
# Plot reconstruction error vs latent dimension.

# Exercise 3: Threshold tuning
# Vary the threshold multiplier (2, 3, 4 standard deviations).
# Plot precision vs recall for different thresholds (ROC-like).

# Exercise 4: Denoising autoencoder
# Add noise to inputs during training: x_noisy = x + 0.1 * randn
# But still reconstruct the clean input. Does this improve robustness?
```

---

## Key Takeaways

1. **Autoencoders learn compressed representations** by encoding-then-decoding input data
2. **The bottleneck forces** the network to learn the most important patterns
3. **Anomaly detection** works because the model cannot reconstruct unusual patterns well
4. **Latent space reveals structure** — soil types cluster naturally in 2D
5. **Non-linear compression** (autoencoder) can capture structure that PCA misses

---

## How This Connects

| ML Concept | Geotechnical Application |
|-----------|------------------------|
| Encoder bottleneck | Extracting key soil classification factors |
| Reconstruction error | Flagging lab errors or unusual soil conditions |
| Latent space | Visualizing site stratigraphy in reduced dimensions |
| Training on normal only | Learning what typical soil looks like |
| Threshold selection | Balancing false alarms vs missed anomalies in QA/QC |